##AI Agent
An AI agent is an AI system that can work toward a goal by deciding what actions to take and then taking those actions(using tools).

A simple way to think about it is:

**AI model + goal + tools + memory/context + decision-making loop = AI agent**

The basic loop is: understand the goal → plan → take an action → observe the result → decide the next action → repeat until done.

##RAG and AI agents(Related) but solve different problems

	#While RAG
*   Give the AI relevant information
  Core capability is (Retrieve + answer)
*   Usually a search/retrieval system as a tool
* Memory-Usually retrieves from a knowledge base
e.g. "What is our company's leave policy?"


#AGENT
*AI system that can take actions
Core capability is to Achieve a goal through multiple steps
* Reason + plan + act
*	Often uses multiple tools
* Can have short/long-term memory
e.g. "Plan my business trip for a day next week.

Goal → Plan → Use tools → Observe results → Decide next action → Repeat

##RAG can be a component inside an AI agent.

n agent might need to know your company's travel policy, so it uses RAG to retrieve that information, then uses other tools to accomplish the task.

In one sentence:
RAG gives an AI access to relevant knowledge; an AI agent gives an AI the ability to decide and take actions.



In [ ]:
pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 4.0 MB/s eta 0:00:00


In [ ]:
import os
from groq import Groq

os.environ["GROQ_API_KEY"] = "ADd Groq Key here"

client = Groq(api_key=os.environ["GROQ_API_KEY"])

print("Groq connected!")


Groq connected!


LLM
 ↓
Does it want a tool?
 ↓
YES
 ↓
Python executes tool
 ↓
Send result back to LLM
 ↓
LLM produces final answer

In [ ]:
## build the actual agent loop

def calculator(expression):
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

print(calculator("25 * 48"))


1200


In [ ]:
##Lets now  Add tool and inform  Groq about it
##We're telling it: "You have this tool available. Decide whether you need it."
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Calculate a mathematical expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "A mathematical expression such as 25 * 48"
                    }
                },
                "required": ["expression"]
            }
        }
    }
]


In [ ]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=messages,
    tools=tools,
    tool_choice="auto"
)

print(response.choices[0].message)


ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning="We need to compute 25 * 48. Could compute manually: 25*48 = 1200? Let's calculate: 25*50 = 1250, minus 25*2 = 50, so 1200. Yes.\n\nWe can also use the calculator tool.", tool_calls=[ChatCompletionMessageToolCall(id='fc_b699fe76-b4e3-4cf3-8e78-5c581e84c4c6', function=Function(arguments='{"expression":"25 * 48"}', name='calculator'), type='function')])


In [ ]:
models = client.models.list()

for model in models.data:
    print(model.id)


openai/gpt-oss-20b
qwen/qwen3.8-27b
openai/gpt-oss-120b
meta-llama/llama-prompt-guard-2-86m
allam-2-7b
openai/gpt-oss-safeguard-20b
whisper-large-v3-turbo
meta-llama/llama-prompt-guard-2-22m
whisper-large-v3
canopylabs/orpheus-arabic-saudi
canopylabs/orpheus-v1-english


In [ ]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=messages,
    tools=tools,
    tool_choice="required"
)

message = response.choices[0].message

print("Assistant response:")
print(message)


Assistant response:
ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning='We need to use the calculator tool.', tool_calls=[ChatCompletionMessageToolCall(id='fc_4c76c8a9-b733-4ce8-9169-998dbe31c403', function=Function(arguments='{"expression":"25 * 48"}', name='calculator'), type='function')])


In [ ]:
print("Tool calls:")
print(message.tool_calls)


Tool calls:
[ChatCompletionMessageToolCall(id='fc_4c76c8a9-b733-4ce8-9169-998dbe31c403', function=Function(arguments='{"expression":"25 * 48"}', name='calculator'), type='function')]


In [ ]:
import json

tool_call = message.tool_calls[0]

function_name = tool_call.function.name
arguments = json.loads(tool_call.function.arguments)

print("AI wants to use:", function_name)
print("Arguments:", arguments)

if function_name == "calculator":
    result = calculator(arguments["expression"])

print("Tool result:", result)


AI wants to use: calculator
Arguments: {'expression': '25 * 48'}
Tool result: 1200


In [ ]:
messages.append(message)

messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": str(result)
})

final_response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=messages,
    tools=tools
)

print(final_response.choices[0].message.content)


25 × 48 = **1,200**
